# Phase 3 – File 3: Backpropagation = Chain Rule on Steroids

**How a 100‑layer network computes $\nabla L$ for *every* weight.**

---

## 1. The core difficulty

A deep net is a **gigantic composition**:

$$ L = (\hat y - y)^2 \; ,\;\; \hat y = w_2 \sigma(w_1 x + b_1) + b_2 $$

The weight $w_1$ appears **deep inside** many nested functions.  
To find $\frac{\partial L}{\partial w_1}$ we must **propagate the error signal backward** through every layer.

---

## 2. The Chain Rule – the universal derivative rule for composition

If $L = f(g(h(w_1)))$ then

$$ \frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial f} \cdot \frac{\partial f f}{\partial g} \cdot \frac{\partial g}{\partial h} \cdot \frac{\partial h}{\partial w_1} $$

Backpropagation **computes each factor locally** and **multiplies** them.

---

## 3. A concrete 1‑hidden‑layer network

```
x  →  z₁ = w₁x + b₁  →  a₁ = σ(z₁)  →  ŷ = w₂a₁ + b₂  →  L = (ŷ - y)²
```

### Forward pass equations

1. $z_1 = w_1 x + b_1$
2. $a_1 = \sigma(z_1) = \frac{1}{1+e^{-z_1}}$
3. $\hat y = w_2 a_1 + b_2$
4. $L = (\hat y - y)^2$

---

## 4. Backward pass – compute every local derivative

| Link | Formula | Explanation |
|------|---------|-------------|
| $\frac{\partial L}{\partial \hat y}$ | $2(\hat y - y)$ | Power rule |
| $\frac{\partial \hat y}{\partial a_1}$ | $w_2$ | Linear |
| $\frac{\partial a_1}{\partial z_1}$ | $a_1(1-a_1)$ | Sigmoid derivative |
| $\frac{\partial z_1}{\partial w_1}$ | $x$ | Linear |

Multiply them:

$$ \frac{\partial L}{\partial w_1} = 2(\hat y-y) \cdot w_2 \cdot a_1(1-a_1) \cdot x $$

---

## 5. Full numeric example (step‑by‑step printout)


In [ ]:
import numpy as np

def sigmoid(z):
    return 1/(1+np.exp(-z))

def sigmoid_prime(a):
    return a*(1-a)

# ---- Data ----
x = 2.0
y = 1.0
w1, b1 = 0.5, 0.1
w2, b2 = -0.3, 0.2

print('=== FORWARD PASS ===')
z1 = w1*x + b1
print(f'z1 = {w1}*{x} + {b1} = {z1:.4f}')
a1 = sigmoid(z1)
print(f'a1 = σ(z1) = {a1:.4f}')
yhat = w2*a1 + b2
print(f'ŷ  = {w2}*{a1:.4f} + {b2} = {yhat:.4f}')
L = (yhat - y)**2
print(f'L  = ({yhat:.4f}-{y})² = {L:.6f}')

print('\n=== BACKWARD PASS (Chain Rule) ===')
dL_dyhat = 2*(yhat - y)
print(f'∂L/∂ŷ  = 2*(ŷ-y) = {dL_dyhat:.4f}')

dyhat_da1 = w2
print(f'∂ŷ/∂a1 = w2 = {dyhat_da1:.4f}')

da1_dz1 = sigmoid_prime(a1)
print(f'∂a1/∂z1 = a1*(1-a1) = {da1_dz1:.4f}')

dz1_dw1 = x
print(f'∂z1/∂w1 = x = {dz1_dw1:.4f}')

dL_dw1 = dL_dyhat * dyhat_da1 * da1_dz1 * dz1_dw1
print(f'\n∂L/∂w1 = {dL_dw1:.6f}')

# ---- One GD step ----
alpha = 0.5
w1_new = w1 - alpha * dL_dw1
print(f'\nUpdate: w1 := {w1} - {alpha}*{dL_dw1:.6f} = {w1_new:.6f}')

=== FORWARD PASS ===
z1 = 0.5*2.0 + 0.1 = 1.1000
a1 = σ(z1) = 0.7503
ŷ  = -0.3*0.7503 + 0.2 = -0.0251
L  = (-0.0251-1.0)² = 1.050785

=== BACKWARD PASS (Chain Rule) ===
∂L/∂ŷ  = 2*(ŷ-y) = -2.0502
∂ŷ/∂a1 = w2 = -0.3000
∂a1/∂z1 = a1*(1-a1) = 0.1874
∂z1/∂w1 = x = 2.0000

∂L/∂w1 = 0.230482

Update: w1 := 0.5 - 0.5*0.230482 = 0.384759


## 6. Generalising to **any depth**

1. **Start at the loss** → compute $\frac{\partial L}{\partial \text{output}}$.
2. **Propagate backward** layer‑by‑layer, multiplying by the **local Jacobian**.
3. **Cache intermediate activations** during the forward pass – they are needed for the Jacobians.

Modern frameworks (PyTorch, TensorFlow) do this **automatically** via `autograd`.

---

## 7. TL;DR

- **Backpropagation = repeated application of the Chain Rule**.
- Each layer contributes a **tiny local derivative**; the product gives the full gradient.
- The algorithm is **O(number of weights)** – extremely efficient.

---

**Next file:** *Optimization* – why plain GD often fails and how Adam rescues us.